In [0]:
%pip install xgboost

In [0]:
# --- 10.1 EXTRAER Y GUARDAR MODELO JSON (Compatibilidad Universal) ---
try:
    import os
    # Navegación: Pipeline -> TransformedTargetRegressor (regressor) -> XGBRegressor (regressor_)
    if hasattr(best_model, 'named_steps'):
        reg_step = best_model.named_steps['regressor']
        if hasattr(reg_step, 'regressor_'):
            xgb_final = reg_step.regressor_
        else:
            xgb_final = reg_step
    else:
        xgb_final = best_model
        
    # Guardar a JSON nativo
    tmp_json = "temp_xgb_model.json"
    xgb_final.save_model(tmp_json)
    with open(tmp_json, "rb") as fjson:
        model_json_bytes = fjson.read()
    os.remove(tmp_json)
    print("✅ Modelo XGBoost convertido a JSON nativo exitosamente.")
except Exception as ej:
    model_json_bytes = None
    print(f"⚠️ No se pudo extraer JSON del modelo: {ej}")

model_bundle = {
    "model":                        None, # Evita error de versión en carga inicial
    "preprocessor_pickle":          pickle.dumps(best_model.named_steps['preprocessor']) if hasattr(best_model, 'named_steps') else None,
    "model_json":                   model_json_bytes,
    "strategy":                     best_strategy,
    "city_stats":                   city_stats,
    "comuna_stats":                 comuna_stats,
    "segment_stats":                segment_stats,
    "micro_segment_stats":          micro_segment_stats,
    "sector_stats":                 sector_stats,
    "sector_reference_stats":       sector_reference_stats,
    "fuente_ratio_stats":           fuente_ratio_stats,
    "fuente_segmento_ratio_stats":  fuente_segmento_ratio_stats,
    "market_meta":                  market_meta,
    "feature_cols":                 cols_numeric_final + cols_categorical + ["texto_completo"],
    "market_features":              market_features,
    "history_feature_cols":         history_feature_cols,
    "amenity_feature_cols":         amenity_feature_cols,
    "history_signal_enabled":       history_signal_enabled,
    "history_repeated_share":       history_repeated_share,
    "ratio_clip_low":               ratio_clip_low  if best_strategy == "residual" else None,
    "ratio_clip_high":              ratio_clip_high if best_strategy == "residual" else None,
    "portal_ops_applied":           bool(portal_weight_map),
    "portal_weight_map":            portal_weight_map,
    "training_source":              training_source,
    "geo_from_gold":                geo_present,
    "trained_at":                   datetime.now(tz=timezone.utc).isoformat(),
    "metrics": {
        "r2": r2, "mae": mae, "mape": mape,
        "train_size": len(X_train), "test_size": len(X_test),
        "history_signal_enabled": history_signal_enabled,
        "portal_ops_applied": bool(portal_weight_map),
    },
    "comparison":  comparison.to_dict(orient="records"),
    "model_type":  "bundle_v8_unified",
}

s3 = boto3.client("s3", aws_access_key_id=config["aws_access_key"], aws_secret_access_key=config["aws_secret_key"])
ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
model_key = f"models/modelo_xgboost_bundle_v8_unified_{ts}.pkl"
# También guardar el JSON por separado en S3
if model_json_bytes:
    s3.put_object(Bucket=BUCKET, Key=model_key.replace(".pkl", ".json"), Body=model_json_bytes)
buf = BytesIO()
pickle.dump(model_bundle, buf)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, model_key)
print(f"\n💾 Bundle guardado: s3://{BUCKET}/{model_key}")
